# Bounded Queues, Backpressure and Little's Law

Companion notebook for the [Producer Consumer and Backpressure](https://codemia.io/courses/concurrency_and_async_programming/cc_producer_consumer_and_backpressure) and [Timeouts, Rate Limits and Bulkheads](https://codemia.io/courses/concurrency_and_async_programming/cc_timeouts_rate_limits_and_bulkheads) lessons on Codemia.

**What you will build**: a discrete event simulator of a producer feeding a pool of workers through a queue. You will watch an unbounded queue grow without limit once the arrival rate passes capacity, see a bounded queue turn that growth into backpressure or rejected work, check Little's law (L = λW) on the simulated numbers, and find the utilization knee where latency takes off. A short real threaded run at the end confirms the simulator.

## Setup

In [ ]:
%pip install -q numpy matplotlib

In [ ]:
import heapq
import queue
import random
import threading
import time

import numpy as np
import matplotlib.pyplot as plt

random.seed(7)
np.random.seed(7)

## 1. The simulator

Requests arrive as a Poisson process with rate `lam` per second. `workers` servers each take an exponential service time with mean `1 / mu`. The queue holds at most `cap` waiting requests (`None` means unbounded). When it is full, `policy` decides what happens:

- `"reject"`: the new request is dropped, the way a load shedder or a full HTTP accept queue behaves.
- `"block"`: the producer stops until a slot frees up, which is backpressure. Arrivals are paused, not lost.

The function returns per request latency, the queue length over time, how many requests were dropped, the end time, and each served request's `(arrival, finish)` pair.

In [ ]:
def simulate(lam, mu, workers=1, cap=None, policy="reject", n=20_000, seed=0):
    rng = random.Random(seed)
    t, free = 0.0, workers
    waiting = []            # arrival times of queued requests, FIFO
    busy = []               # heap of finish times
    spans, qlen, dropped, blocked_until = [], [], 0, 0.0
    head = 0
    for _ in range(n):
        t = max(t + rng.expovariate(lam), blocked_until)
        while busy and busy[0] <= t:           # finish everything done before this arrival
            done = heapq.heappop(busy)
            if head < len(waiting):
                a = waiting[head]; head += 1
                fin = done + rng.expovariate(mu)
                spans.append((a, fin))
                heapq.heappush(busy, fin)
            else:
                free += 1
        depth = len(waiting) - head
        if free:
            free -= 1
            fin = t + rng.expovariate(mu)
            spans.append((t, fin))
            heapq.heappush(busy, fin)
        elif cap is None or depth < cap:
            waiting.append(t)
        elif policy == "reject":
            dropped += 1
        else:  # block: the producer waits for the next completion, then enqueues
            done = heapq.heappop(busy)
            a = waiting[head]; head += 1
            fin = done + rng.expovariate(mu)
            spans.append((a, fin))
            heapq.heappush(busy, fin)
            waiting.append(done)
            blocked_until = t = done
        qlen.append((t, len(waiting) - head))
    lat = np.array([f - a for a, f in spans])
    return lat, qlen, dropped, t, spans

## 2. Past capacity, an unbounded queue only grows

Capacity is `workers * mu`. Below it, the queue hovers near zero. Above it, the backlog grows by roughly `lam - capacity` every second and never drains, and every new request waits behind all of it. Memory and latency both grow without limit, which is how an unbounded queue turns a burst into an outage.

In [ ]:
plt.figure(figsize=(7, 3.2))
for lam in [80, 95, 110]:
    _, qlen, _, _, _ = simulate(lam, mu=100, n=20_000)
    ts, qs = zip(*qlen)
    plt.plot(ts, qs, label=f"arrivals {lam}/s (capacity 100/s)")
plt.xlabel("time (s)")
plt.ylabel("queue length")
plt.title("Unbounded queue under and over capacity")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 3. Bounding the queue: reject or push back

Now cap the queue at 50 at 110 requests per second. With `reject`, latency stays bounded and the excess is dropped, so callers get a fast error they can retry elsewhere. With `block`, nothing is lost but the producer slows to the consumer's pace. That is backpressure: the overload is pushed upstream instead of stored.

In [ ]:
for cap, policy in [(None, "reject"), (50, "reject"), (50, "block")]:
    lat, _, dropped, end, _ = simulate(110, mu=100, cap=cap, policy=policy, n=20_000)
    print(f"cap={str(cap):4s} {policy:6s}  p50 {np.percentile(lat, 50) * 1000:7.1f} ms  "
          f"p99 {np.percentile(lat, 99) * 1000:7.1f} ms  dropped {dropped:5d}  "
          f"throughput {len(lat) / end:6.1f}/s")

## 4. Check Little's law

Little's law says the average number in the system equals the arrival rate times the average time spent: `L = λW`. It holds for any stable queue, whatever the distributions. Here we measure L directly: sweep through every arrival and departure, track how many requests are in the system, and average that count over time. Then we compare it with the offered rate λ times the measured mean latency W.

In [ ]:
def littles_check(lam, mu, workers):
    lat, _, _, _, spans = simulate(lam, mu, workers=workers, n=50_000, seed=3)
    events = sorted([(a, 1) for a, _ in spans] + [(f, -1) for _, f in spans])
    area, n_in, last = 0.0, 0, events[0][0]
    for t, step in events:                 # integrate N(t), the number in the system
        area += n_in * (t - last)
        n_in += step
        last = t
    L = area / (events[-1][0] - events[0][0])
    return L, lam * lat.mean()

for lam, mu, w in [(50, 100, 1), (90, 100, 1), (350, 100, 4)]:
    L, lw = littles_check(lam, mu, w)
    print(f"lam={lam:3d} workers={w}  L={L:6.2f}  lam*W={lw:6.2f}")

## 5. The utilization knee

Average latency for a single server queue grows like `1 / (1 - ρ)` where ρ is utilization. It looks flat until about 70 to 80 percent, then climbs steeply. That is why pools are sized to run well below 100 percent, and why a small rise in traffic near capacity causes a large rise in latency.

In [ ]:
rhos = np.linspace(0.1, 0.97, 18)
sim = [simulate(r * 100, 100, n=40_000, seed=5)[0].mean() * 1000 for r in rhos]
theory = [1000 / (100 * (1 - r)) for r in rhos]   # M/M/1: W = 1 / (mu - lam)
plt.figure(figsize=(6.5, 3.2))
plt.plot(rhos, sim, "o", label="simulated")
plt.plot(rhos, theory, label="M/M/1 theory")
plt.xlabel("utilization ρ")
plt.ylabel("mean latency (ms)")
plt.title("Latency explodes as utilization nears 1")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 6. The real thing: threads and `queue.Queue(maxsize)`

`queue.Queue(maxsize=k)` gives you blocking backpressure for free: `put` waits while the queue is full. Here a producer tries to submit 200 items every 2 ms while two consumers each need 10 ms per item, so capacity is 200 per second against 500 offered. With `maxsize=0` (unbounded) the producer finishes quickly and the backlog piles up; with `maxsize=10` the producer is slowed to the consumers' pace and the queue never exceeds 10.

In [ ]:
def threaded_run(maxsize, items=200, gap=0.002, service=0.01, consumers=2):
    q = queue.Queue(maxsize=maxsize)
    peak = 0
    def consumer():
        while True:
            item = q.get()
            if item is None:
                return
            time.sleep(service)
    threads = [threading.Thread(target=consumer) for _ in range(consumers)]
    for th in threads:
        th.start()
    t0 = time.perf_counter()
    for i in range(items):
        q.put(i)
        peak = max(peak, q.qsize())
        time.sleep(gap)
    producer_time = time.perf_counter() - t0
    for _ in threads:
        q.put(None)
    for th in threads:
        th.join()
    return producer_time, peak, time.perf_counter() - t0

for m in [0, 10]:
    p, peak, total = threaded_run(m)
    print(f"maxsize={m:2d}  producer done in {p:.2f}s  peak queue {peak:3d}  all done in {total:.2f}s")

## 7. Exercises

1. Size a pool with Little's law: at 400 requests per second and a 50 ms mean service time, how many workers keep utilization near 70 percent? Check your answer with `simulate`.
2. Add a deadline to each request in `simulate` and drop work that has already waited longer than its deadline when it reaches a worker. How does goodput compare with plain `reject`?
3. Serve the queue LIFO instead of FIFO under overload, as adaptive LIFO does, and compare p50 and p99. Why does p50 improve?
4. Replace the exponential service time with a heavy tailed one (`rng.paretovariate(1.5)` scaled to the same mean) and see how far the knee moves left.